In [1]:
# Runtime evidence: this cell checks the environment actually used by the kernel.
import sys as _shape_sys, json as _shape_json, importlib.util as _shape_util
print("SHAPE_RUNTIME_PROVENANCE=" + _shape_json.dumps({
    "python_executable": _shape_sys.executable,
    "python_version": _shape_sys.version.split()[0],
    "shape_lab_origin": _shape_util.find_spec("shape_lab").origin
}))
del _shape_sys, _shape_json, _shape_util


SHAPE_RUNTIME_PROVENANCE={"python_executable": "/opt/pyvenv/bin/python", "python_version": "3.13.5", "shape_lab_origin": "/tmp/shape-course-q90rdcpz/course/src/shape_lab/__init__.py"}


# wine: worked coding answers

Read the [case lesson](../lessons/wine.md). Every reference function below has executed checks. Compare only after attempting.

These checks verify limited function contracts, not your scientific interpretation.

In [2]:
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic('matplotlib','inline')
import numpy as np
import pandas as pd

## wine.C1 — Use fitted scaling

Standardize a finite 2-D matrix using supplied training means and positive scales. Do not calculate new means.

Predict the check output before writing your function.

In [3]:
def apply_scale(x,mean,scale):
    a,m,s=np.asarray(x,float),np.asarray(mean,float),np.asarray(scale,float)
    if a.ndim!=2 or m.shape!=(a.shape[1],) or s.shape!=m.shape or np.any(s<=0): raise ValueError('Incompatible shape or scale')
    return (a-m)/s

In [4]:
np.testing.assert_allclose(apply_scale([[101,5]],[2,5],[1,1]),[[99,0]])
np.testing.assert_allclose(apply_scale([[2,4]],[1,2],[1,2]),[[1,1]])
print("wine.C1 checks passed")

wine.C1 checks passed


**Why it works:** The training transformation is frozen. Recalculating it on the test query would change the model definition.

## wine.C2 — Choose nearby reference points

Return indices of the k nearest reference points to a query using Euclidean distance; preserve input order on ties.

Predict the check output before writing your function.

In [5]:
def nearest_reference(query,reference,k):
    q,a=np.asarray(query,float),np.asarray(reference,float)
    if a.ndim!=2 or q.shape!=(a.shape[1],) or not 1<=k<=len(a): raise ValueError('Bad neighbor request')
    return np.argsort(np.linalg.norm(a-q,axis=1),kind='stable')[:k]

In [6]:
np.testing.assert_array_equal(nearest_reference([0,0],[[4,0],[1,0],[2,0]],2),[1,2])
np.testing.assert_array_equal(nearest_reference([0],[[1],[-1]],2),[0,1])
print("wine.C2 checks passed")

wine.C2 checks passed


**Why it works:** Distance selects neighbors; sorting indices preserves the link back to the training rows. The tie convention differs explicitly from the production teaching descriptor’s coordinate-based tie rule.

## wine.C3 — Filter finite lifetimes

Given an n-by-2 birth/death array, return death-birth for finite positive-lifetime intervals only.

Predict the check output before writing your function.

In [7]:
def finite_lifetimes(diagram):
    a=np.asarray(diagram,float).reshape(-1,2)
    valid=np.isfinite(a).all(axis=1)&(a[:,1]>a[:,0])
    return a[valid,1]-a[valid,0]

In [8]:
np.testing.assert_allclose(finite_lifetimes([[0,1],[2,2],[0,np.inf],[3,5]]),[1,2])
assert finite_lifetimes([]).size==0
print("wine.C3 checks passed")

wine.C3 checks passed


**Why it works:** Essential classes need separate handling. Silently averaging infinity is not a meaningful finite persistence statistic.